In [8]:
from pathlib import Path
import json
import pymupdf

In [9]:
PAPERS_DIR = Path("../data/papers")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)   # create the folder if
PAPERS_DIR.mkdir(parents=True, exist_ok=True)   # create the folder if missing

In [10]:
pdf_files = sorted(PAPERS_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files")
for f in pdf_files:
    print(f" - {f.name}")

Found 2 PDF files
 - 1. Overview of Computer Security_032115.pdf
 - vincent proposal.pdf


In [13]:
reader = pymupdf.open(pdf_files[0])          # open the first PDF
print("Number of pages:", reader.page_count)

first_page = reader[0].get_text()
print("\n--- First 600 characters of page 1 ---")
print(first_page[:600])

first_page = reader.pages[0].extract_text()   # get the text of page 1
print("\n--- First 600 characters of page 1 ---")
print(first_page[:600])

Number of pages: 42

--- First 600 characters of page 1 ---
1. Overview of Computer 
Security 



TypeError: 'method' object is not subscriptable

In [14]:
pages = []

for pdf_path in pdf_files:
    document = pymupdf.open(pdf_path)
    for page_number, page in enumerate(document, start=1):
        text = page.get_text()
        pages.append({
            "source": pdf_path.name,
            "page_number": page_number,
            "text": text
        })
    document.close()

print(f"Extracted text from {len(pages)} pages")
print("Example page record keys:", pages[0].keys() if pages else "No pages found")

Extracted text from 99 pages
Example page record keys: dict_keys(['source', 'page_number', 'text'])


In [17]:
import re

def clean_text(text):
    text = re.sub(r"-\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("BEFORE:", repr(first_page[:200]))
print ()
print("AFTER:", repr(clean_text(first_page[:200])))

for p in pages:
    p["text"] = clean_text(p["text"])

BEFORE: '1. Overview of Computer \nSecurity \n'

AFTER: '1. Overview of Computer Security'


In [18]:
def chunk_text(text, chunk_size=800, overlap=150):
    assert overlap < chunk_size, "Overlap must be smaller than chunk size"

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

#Demo 
demo = "0123456789" * 5  #a  50-character string
for c in chunk_text(demo, chunk_size=20, overlap=5):
    print(c)

01234567890123456789
56789012345678901234
01234567890123456789
56789


In [21]:
CHUNK_SIZE = 800
OVERLAP = 150

chunks = []
for p in pages:
    for piece in chunk_text(p["text"], CHUNK_SIZE, OVERLAP):
        if len(piece.strip()) > 50:
            chunks.append({
                "id": len(chunks),
                "text": piece,
                "source": p["source"],
                "page": p["page_number"]
            })

print(f"Created {len(chunks)} chunks")
print("\nExample chunk:")
print(chunks[0])

Created 188 chunks

Example chunk:
{'id': 0, 'text': 'Roadmap • Computer Security Concepts • Threats, Attacks, and Assets • Malicious Software Overview • Viruses, Worms, and Bots • Rootkits', 'source': '1. Overview of Computer Security_032115.pdf', 'page': 2}


In [22]:
for size in [300, 500, 800, 1200, 2000]:
    total = 0
    for p in pages:
        total += len([c for c in chunk_text(p["text"], size, size // 5) if len(c.strip()) > 50])
    print(f"chunk_size={size:>5} -> {total} chunks")

chunk_size=  300 -> 417 chunks
chunk_size=  500 -> 272 chunks
chunk_size=  800 -> 195 chunks
chunk_size= 1200 -> 150 chunks
chunk_size= 2000 -> 116 chunks


In [24]:
output_path = PROCESSED_DIR / "chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(chunks)} chunks to {output_path}")

Saved 188 chunks to ../data/processed/chunks.json
